In [ ]:
print("SIH26162 Started!")

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd() / "SIH26162"

df = pd.read_csv(BASE / "data" / "sample_fire_data.csv")
sites = pd.read_csv(BASE / "data" / "industrial_sites.csv")

print("Fire data:", len(df))
print("Industrial sites:", len(sites))

In [ ]:
import os
print(os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium

print("SIH26162 Environment Ready!")

In [ ]:
import pandas as pd

df = pd.read_csv("SIH26162/data/sample_fire_data.csv")

print(df)

In [ ]:
print("Number of hotspots:", len(df))

In [ ]:
import folium

# India map
m = folium.Map(
    location=[22.5, 80],
    zoom_start=5
)

# Add thermal hotspots
for _, row in df.iterrows():

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=7,
        popup=(
            f"Brightness: {row['brightness']}<br>"
            f"Confidence: {row['confidence']}%<br>"
            f"Date: {row['acq_date']}"
        ),
        fill=True
    ).add_to(m)

# Display map
m

In [ ]:
# Basic statistics
print("Total hotspots:", len(df))
print("Average brightness:", df["brightness"].mean())
print("Average confidence:", df["confidence"].mean())
print("Highest brightness:", df["brightness"].max())
print("Highest confidence:", df["confidence"].max())

In [ ]:
df_sorted = df.sort_values("confidence", ascending=False)

df_sorted

In [ ]:
high_conf = df[df["confidence"] >= 80]

print("High confidence hotspots:", len(high_conf))
high_conf

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib

In [ ]:
import matplotlib
print(matplotlib.__version__)

In [ ]:
import pandas as pd

df = pd.read_csv("SIH26162/data/sample_fire_data.csv")

print(df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.scatter(
    df["longitude"],
    df["latitude"],
    s=df["brightness"] * 2,
    alpha=0.7
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Thermal Hotspot Distribution")

plt.show()

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("SIH26162/data/sample_fire_data.csv")

df["acq_date"] = pd.to_datetime(df["acq_date"])

print(df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.scatter(
    df["longitude"],
    df["latitude"],
    s=df["brightness"] * 2,
    alpha=0.7
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Thermal Hotspot Distribution")

plt.show()

In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("SIH26162/data/sample_fire_data.csv")

df["acq_date"] = pd.to_datetime(df["acq_date"])

print(df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.scatter(
    df["longitude"],
    df["latitude"],
    s=df["brightness"] * 2,
    alpha=0.7
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Thermal Hotspot Distribution")

plt.show()

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
persistence = ( #persistence ka mtlb h ki aag kitni der tak aur kitni baar ek hi jagah per tikti hai
    df.groupby(
        [
            df["latitude"].round(3),#round() upto 3 decimal place
            df["longitude"].round(3)
        ]
    )
    .agg(    #dbms jaisa aggregate functions ka use hua h
        detections=("acq_date", "count"),
        avg_brightness=("brightness", "mean"),
        avg_confidence=("confidence", "mean"),
        total_frp=("frp", "sum")
    )
    .reset_index()
)

persistence

In [ ]:
persistence["persistent"] = persistence["detections"] >= 3 #persistent ek naya column add krenge jo detections coln pe work krega
   #agar detection>3 then true otherwise false

persistence

In [ ]:
from math import radians, sin, cos, sqrt, atan2
#yw simply different approaches se distance_km banata hai

def distance_km(lat1, lon1, lat2, lon2):
    R = 6371

    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = (
        sin(dlat / 2) ** 2
        + cos(radians(lat1))
        * cos(radians(lat2))
        * sin(dlon / 2) ** 2
    )

    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

In [ ]:
sites = pd.read_csv("SIH26162/data/industrial_sites.csv")

print(sites)

In [ ]:
def nearest_industrial_site(row):#ye bhi aag ke point ke liye , yeh pta lagana ki uske sabse nazdeek kaunsi industry hai wo kitne km dur h

    distances = sites.apply(
        lambda site: distance_km( #woks on industrial_sites.csv
            row["latitude"],
            row["longitude"],
            site["latitude"],
            site["longitude"]
            # result ki ye kitne km dur h
        ),
        axis=1
    )

    idx = distances.idxmin()#idxmin() means sbse chhoti value minimum distance  kahan h

    return pd.Series({
        "nearest_site": sites.loc[idx, "site_name"],
        "site_type": sites.loc[idx, "site_type"],
        "distance_km": distances.loc[idx]
    })

In [ ]:
nearest = df.apply(nearest_industrial_site, axis=1)# sbse pass wala industry ka naam dega 

df = pd.concat([df, nearest], axis=1)# concatination performed

df #take 16 min to execute

In [ ]:
def classify_source(row):  #if else perform hua hai

    if row["distance_km"] <= 2 and row["confidence"] >= 80:
        return "Industrial Thermal Source"

    elif row["brightness"] >= 350 and row["confidence"] >= 80:
        return "High Intensity Fire"

    else:
        return "Other Thermal Anomaly"

In [ ]:
df["source_class"] = df.apply(classify_source, axis=1) #source con add kiya upper wale classification ke basis pe

df[
    [
        "latitude",
        "longitude",
        "brightness",
        "confidence",
        "distance_km",
        "nearest_site",
        "source_class"
    ]
]

In [ ]:
import folium #showing map

m = folium.Map(
    location=[23, 82],
    zoom_start=5
)

for _, row in df.iterrows():

    popup = f"""
    <b>Classification:</b> {row['source_class']}<br>
    <b>Brightness:</b> {row['brightness']}<br>
    <b>Confidence:</b> {row['confidence']}%<br>
    <b>Nearest Industry:</b> {row['nearest_site']}<br>
    <b>Distance:</b> {row['distance_km']:.2f} km
    """

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=7,
        popup=popup,
        fill=True
    ).add_to(m)

m

In [ ]:
m.save("../SIH26162_final_map.html")

print("Final GIS map generated!")

In [ ]:
import pandas as pd
import numpy as np
import folium

from math import radians, sin, cos, sqrt, atan2

print("Libraries loaded successfully!")

In [ ]:
df = pd.read_csv("../data/sample_fire_data.csv")

#ERROR

df["acq_date"] = pd.to_datetime(df["acq_date"])

df = df.dropna(
    subset=[
        "latitude",
        "longitude",
        "brightness",
        "confidence",
        "frp"
    ]
).copy()

print("Total detections:", len(df))
print(df.columns.tolist())

In [ ]:
import os

print("Current notebook folder:")
print(os.getcwd())

In [ ]:
import os

print("\nFiles/folders here:")
print(os.listdir())

In [ ]:
import pandas as pd
import numpy as np
import folium

from math import radians, sin, cos, sqrt, atan2

print("Libraries loaded successfully!")

In [ ]:
from pathlib import Path #file ke path ko dhoondhna

# Find project folder automatically
current = Path.cwd()

print("Current working directory:")
print(current)

# Try common locations
possible_paths = [
    current / "../data/sample_fire_data.csv",
    current / "data/sample_fire_data.csv",
    current / "SIH26162/data/sample_fire_data.csv"
]

fire_file = None

for path in possible_paths:
    path = path.resolve()
    if path.exists():
        fire_file = path
        break

if fire_file is None:
    raise FileNotFoundError(
        "sample_fire_data.csv nahi mila. "
        "Check karo ki file data folder ke andar hai."
    )

print("Using file:")
print(fire_file)

df = pd.read_csv(fire_file) #yahah tk me file mil gyi h toh pandas usko ek dataframe mein load kr lega

df["acq_date"] = pd.to_datetime(df["acq_date"])# csv me date string me hoti hai pandas isko dateformat me change kr dega

df = df.dropna(# cleaning 
    subset=[
        "latitude",
        "longitude",
        "brightness",
        "confidence",
        "frp"
    ]
).copy()

print("Total detections:", len(df))
print("\nColumns:")
print(df.columns.tolist())#ye btaega ki table me kaun kaun sa column hai

display(df)

In [ ]:
df["lat_group"] = df["latitude"].round(3)#adding new cols
df["lon_group"] = df["longitude"].round(3)

persistence = (
    df.groupby(
        ["lat_group", "lon_group"]
    )
    .agg( #fir aggregate functions
        detections=("acq_date", "nunique"),
        avg_brightness=("brightness", "mean"),
        avg_confidence=("confidence", "mean"),
        total_frp=("frp", "sum")
    )
    .reset_index()
)

# Persistent if detected on at least 3 different dates
persistence["persistent"] = (
    persistence["detections"] >= 3
)

print("Persistence table:")
display(persistence)

In [ ]:
print("Persistence columns:")
print(persistence.columns.tolist())#check peristance cols

print("\nPersistent locations:")

display( #display only important cols
    persistence[
        [
            "lat_group",
            "lon_group",
            "detections",
            "persistent"
        ]
    ]
)

In [ ]:
possible_site_paths = [#list of current files
    current / "../data/industrial_sites.csv",
    current / "data/industrial_sites.csv",
    current / "SIH26162/data/industrial_sites.csv"
]

site_file = None #empty variable kyunki same nhi pta ki file kahah hai

for path in possible_site_paths:
    path = path.resolve()#path ko absolute bna deta hai
    if path.exists():
        site_file = path
        break

if site_file is None:
    raise FileNotFoundError(
        "industrial_sites.csv nahi mila. "
        "Check karo ki file data folder ke andar hai."
    )

print("Using file:")
print(site_file)

sites = pd.read_csv(site_file)#ab file mil gyi hogi to sites var me store ho jaega

print("Industrial sites:")
display(sites)

In [ ]:
def distance_km(lat1, lon1, lat2, lon2):# finding distance in km by comparing both csv files  samjha 

    R = 6371.0

    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = (
        sin(dlat / 2) ** 2
        +
        cos(radians(lat1))
        * cos(radians(lat2))
        * sin(dlon / 2) ** 2
    )

    return R * 2 * atan2(
        sqrt(a),
        sqrt(1 - a)
    )

In [ ]:
def nearest_industrial_site(row):#function jo every single row pe work krega 

    distances = sites.apply( #sites pe apply kiya
        lambda site: distance_km(
            row["latitude"],
            row["longitude"],
            site["latitude"],
            site["longitude"]
        ),
        axis=1
    )

    nearest_index = distances.idxmin()# minimum distance

    return pd.Series({
        "nearest_site": sites.loc[
            nearest_index,
            "site_name"
        ],

        "site_type": sites.loc[
            nearest_index,
            "site_type"
        ],

        "distance_km": distances.loc[
            nearest_index
        ]
    })

In [ ]:
nearest = df.apply(
    nearest_industrial_site,# use of above function
    axis=1
)

df = pd.concat( # concatination
    [df, nearest],
    axis=1
)

display(
    df[
        [
            "latitude",
            "longitude",
            "nearest_site",
            "site_type",
            "distance_km"
        ]
    ]
)

In [ ]:
df = df.merge(#merging
    persistence[
        [
            "lat_group",
            "lon_group",
            "detections",
            "avg_brightness",
            "avg_confidence",
            "total_frp",
            "persistent"
        ]
    ],
    on=[
        "lat_group",
        "lon_group"
    ],
    how="left"
)

print("Merge successful!")

display(
    df[
        [
            "latitude",
            "longitude",
            "detections",
            "persistent",
            "avg_brightness",
            "total_frp"
        ]
    ]
)

In [ ]:
def classify_source(row):# classify on each row

    # Persistent + near industry + high confidence
    if (
        row["persistent"]
        and row["distance_km"] <= 2
        and row["confidence"] >= 80
    ):
        return "Persistent Industrial Source"

    # High intensity thermal anomaly
    elif (
        row["brightness"] >= 350
        and row["confidence"] >= 80
    ):
        return "High Intensity Fire"

    # Persistent but not near industry
    elif row["persistent"]:
        return "Persistent Non-Industrial Thermal Source"

    # Everything else
    else:
        return "Other Thermal Anomaly"

In [ ]:
df["source_class"] = df.apply(#stored in source_class
    classify_source,
    axis=1
)

display(
    df[
        [
            "latitude",
            "longitude",
            "detections",
            "persistent",
            "distance_km",
            "nearest_site",
            "source_class"
        ]
    ]
)

print("\nSource classification counts:")
print(df["source_class"].value_counts())

In [ ]:
def calculate_risk(row):# conditions to calculate risk see it 
    #vvi coln

    score = 0

    # Confidence
    if row["confidence"] >= 90:
        score += 25
    elif row["confidence"] >= 80:
        score += 15

    # Brightness
    if row["brightness"] >= 370:
        score += 25
    elif row["brightness"] >= 350:
        score += 15

    # Persistence
    if row["detections"] >= 5:
        score += 30
    elif row["detections"] >= 3:
        score += 20

    # Industrial proximity
    if row["distance_km"] <= 1:
        score += 20
    elif row["distance_km"] <= 2:
        score += 10

    return min(score, 100)

In [ ]:
df["risk_score"] = df.apply(# adding new coln risk scre
    calculate_risk,
    axis=1
)

display(
    df[
        [
            "latitude",
            "longitude",
            "source_class",
            "risk_score"
        ]
    ]
)

In [ ]:
def risk_category(score):# based on score risk category

    if score >= 75:
        return "High"

    elif score >= 50:
        return "Medium"

    else:
        return "Low"


df["risk_level"] = df[#adding risk level coln based on risk score
    "risk_score"
].apply(risk_category)

display(
    df[
        [
            "source_class",
            "risk_score",
            "risk_level"
        ]
    ]
)

In [ ]:
final_columns = [# feature engg krne ke baad final cols
    "latitude",
    "longitude",
    "brightness",
    "confidence",
    "acq_date",
    "acq_time",
    "frp",
    "detections",
    "persistent",
    "avg_brightness",
    "avg_confidence",
    "total_frp",
    "nearest_site",
    "site_type",
    "distance_km",
    "source_class",
    "risk_score",
    "risk_level"
]

final_df = df[final_columns].copy()

display(final_df)

In [ ]:
#YE FILE HI FRONTEND ME JAEGI 

output_file = fire_file.parent / "processed_thermal_sources.csv"
#new output file is made after comparing and processing both csv files

final_df.to_csv(
    output_file,
    index=False
)

print("✅ Processed dataset saved:")
print(output_file)

In [ ]:
def marker_color(source_class):  #coloring scheme

    if source_class == "Persistent Industrial Source":
        return "red"

    elif source_class == "High Intensity Fire":
        return "orange"

    elif source_class == "Persistent Non-Industrial Thermal Source":
        return "blue"

    else:
        return "gray"

In [ ]:
m = folium.Map( #using coloring scheme on map
    location=[23, 82],
    zoom_start=5
)

for _, row in final_df.iterrows():

    popup = f"""
    <b>Classification:</b>
    {row['source_class']}<br>

    <b>Risk:</b>
    {row['risk_level']}
    ({row['risk_score']}/100)<br>

    <b>Brightness:</b>
    {row['brightness']}<br>

    <b>Confidence:</b>
    {row['confidence']}%<br>

    <b>Detections:</b>
    {row['detections']}<br>

    <b>Nearest Industry:</b>
    {row['nearest_site']}<br>

    <b>Industry Type:</b>
    {row['site_type']}<br>

    <b>Distance:</b>
    {row['distance_km']:.2f} km
    """

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],

        radius=8,

        popup=popup,

        color=marker_color(
            row["source_class"]
        ),

        fill=True,

        fill_opacity=0.7

    ).add_to(m)

m

In [ ]:
map_file = fire_file.parent.parent / "SIH26162_final_map.html"
# generation of map using html

m.save(map_file)

print("✅ Final GIS map generated!")
print(map_file)

In [ ]:
print("========== SIH26162 SUMMARY ==========")

print(
    "Total Thermal Detections:",
    len(final_df)
)

print(
    "Persistent Detections:",
    int(final_df["persistent"].sum())
)

print(
    "Persistent Industrial Sources:",
    int(
        (
            final_df["source_class"]
            == "Persistent Industrial Source"
        ).sum()
    )
)

print(
    "High Intensity Fires:",
    int(
        (
            final_df["source_class"]
            == "High Intensity Fire"
        ).sum()
    )
)

print(
    "High Risk Sources:",
    int(
        (
            final_df["risk_level"]
            == "High"
        ).sum()
    )
)

print("========================================")

In [ ]:
import sys 
print(sys.executable)

In [ ]:
import pandas as pd
import numpy as np

base = r"C:\Users\Hp\OneDrive\Desktop\SIH26162"

fires = pd.read_csv(base + r"\data\sample_fire_data.csv")
land = pd.read_csv(base + r"\data\land_cover.csv")
sites = pd.read_csv(base + r"\data\industrial_sites.csv")

print("Fire data:", fires.shape)
print("Land-cover:", land.shape)
print("Industrial sites:", sites.shape)

In [ ]:
print("Fire columns:")
print(fires.columns.tolist())

print("\nLand-cover columns:")
print(land.columns.tolist())

print("\nIndustrial columns:")
print(sites.columns.tolist())

In [ ]:
def nearest_landcover(row):
    distances = (
        (land["latitude"] - row["latitude"]) ** 2 +
        (land["longitude"] - row["longitude"]) ** 2
    )

    idx = distances.idxmin()

    return land.loc[idx, ["land_cover", "vegetation_index"]]


fires[["land_cover", "vegetation_index"]] = fires.apply(
    nearest_landcover,
    axis=1,
    result_type="expand"
)

print(fires.head())

In [ ]:
print(fires["land_cover"].value_counts())

print("\nVegetation index:")
print(fires["vegetation_index"].describe())

In [ ]:
fires.to_csv(
    r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data\fire_landcover_data.csv",
    index=False
)

print("File saved successfully!")

In [ ]:
import pandas as pd
import numpy as np

sites = pd.read_csv(
    r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data\industrial_sites.csv"
)

print("Industrial sites:", sites.shape)
print(sites.columns.tolist())
print(sites.head())

In [ ]:
def nearest_industry_distance(row):
    distances = np.sqrt(
        (sites["latitude"] - row["latitude"]) ** 2 +
        (sites["longitude"] - row["longitude"]) ** 2
    )

    return distances.min()


fires["distance_to_industry"] = fires.apply(
    nearest_industry_distance,
    axis=1
)

print(fires[[
    "latitude",
    "longitude",
    "distance_to_industry"
]].head())

In [ ]:
fires["distance_to_industry_km"] = (
    fires["distance_to_industry"] * 111
)

fires.drop(
    columns=["distance_to_industry"],
    inplace=True
)

print(
    fires["distance_to_industry_km"].describe()
)

In [ ]:
print(fires.columns.tolist())

In [ ]:
fires["location_key"] = (
    fires["latitude"].round(2).astype(str)
    + "_"
    + fires["longitude"].round(2).astype(str)
)

fires["persistence_count"] = (
    fires.groupby("location_key")["location_key"]
    .transform("count")
)

print(
    fires[
        ["latitude", "longitude", "persistence_count"]
    ].head(10)
)

In [ ]:
print(fires["land_cover"].value_counts())

In [ ]:
land_cover_mapping = {
    "industrial": 0,
    "built_up": 1,
    "forest": 2,
    "agriculture": 3,
    "mining": 4,
    "water": 5
}

fires["land_cover_code"] = fires["land_cover"].map(
    land_cover_mapping
)

print(
    fires[
        ["land_cover", "land_cover_code"]
    ].head(10)
)

In [ ]:
print(fires.columns.tolist())

In [ ]:
features = [
    "latitude",
    "longitude",
    "vegetation_index",
    "distance_to_industry_km",
    "persistence_count",
    "land_cover_code"
]

print(fires[features].head())

In [ ]:
print(fires[features].isnull().sum())

In [ ]:
print(fires.columns.tolist())



print(fires["land_cover"].value_counts())



print(sites.columns.tolist())

In [ ]:
features = [
    "brightness",
    "confidence",
    "frp",
    "vegetation_index",
    "distance_to_industry_km",
    "persistence_count",
    "land_cover_code"
]

X = fires[features].copy()

print(X.head())
print("\nMissing values:")
print(X.isnull().sum())

In [ ]:
def classify_fire(row):

    # Industrial area + close to industrial facility
    if (
        row["land_cover"] == "industrial"
        and row["distance_to_industry_km"] <= 5
    ):
        if row["persistence_count"] >= 3:
            return "Persistent Industrial Thermal Source"
        else:
            return "Industrial Fire"

    # Forest + strong thermal signal
    elif (
        row["land_cover"] == "forest"
        and row["frp"] >= 20
    ):
        return "Wildfire"

    # Agriculture + thermal activity
    elif (
        row["land_cover"] == "agriculture"
        and row["frp"] >= 10
    ):
        return "Agricultural Burning"

    else:
        return "Other Thermal Source"


fires["source_class"] = fires.apply(
    classify_fire,
    axis=1
)

print(fires["source_class"].value_counts())

In [ ]:
print(
    fires[
        [
            "brightness",
            "frp",
            "land_cover",
            "distance_to_industry_km",
            "persistence_count",
            "source_class"
        ]
    ].head(20)
)

In [ ]:
def calculate_confidence(row):

    score = 0

    if row["confidence"] >= 80:
        score += 0.35
    elif row["confidence"] >= 50:
        score += 0.20

    if row["frp"] >= 20:
        score += 0.25
    elif row["frp"] >= 10:
        score += 0.15

    if row["distance_to_industry_km"] <= 5:
        score += 0.20

    if row["persistence_count"] >= 3:
        score += 0.20

    return min(score, 1.0)


fires["classification_confidence"] = fires.apply(
    calculate_confidence,
    axis=1
)

print(
    fires[
        ["source_class", "classification_confidence"]
    ].head(20)
)

In [ ]:
fires["confidence_percent"] = (
    fires["classification_confidence"] * 100
).round(2)

print(
    fires[
        ["source_class", "confidence_percent"]
    ].head(20)
)

In [ ]:
fires["risk_score"] = (
    fires["frp"].rank(pct=True) * 40
    + fires["persistence_count"].rank(pct=True) * 30
    + (1 - fires["distance_to_industry_km"].rank(pct=True)) * 30
)

fires["risk_score"] = fires["risk_score"].round(2)

print(
    fires[
        [
            "frp",
            "persistence_count",
            "distance_to_industry_km",
            "risk_score"
        ]
    ].head(20)
)

In [ ]:
output_path = r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data\classified_fire_data.csv"

fires.to_csv(
    output_path,
    index=False
)

print("Final dataset saved successfully!")
print(output_path)

In [ ]:
print(fires.columns.tolist())

nearest industry add krne ki kosis neeche h

In [ ]:
import pandas as pd

fires_path = "../data/sample_fire_data.csv"
sites_path = "../data/industrial_sites.csv"

fires = pd.read_csv(fires_path)
sites = pd.read_csv(sites_path)

print("Fires:", fires.shape)
print("Industrial sites:", sites.shape)

print(sites[['site_name', 'site_type', 'latitude', 'longitude']].head())

In [ ]:
import os

# 1. Pehle check karo ki current folder kya hai
print("Current folder:", os.getcwd())

# 2. Absolute path use karke save karo (apne system ka sahi path daalo)
save_path = r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data\classified_fire_data.csv"

# 3. Check karo ki folder exist karta hai ya nahi
if os.path.exists(os.path.dirname(save_path)):
    fires.to_csv(save_path, index=False)
    print(f"✅ File save ho gayi: {save_path}")
else:
    print(f"❌ Folder nahi mila: {os.path.dirname(save_path)}")

In [ ]:
import numpy as np

nearest_idx = []

for _, fire in fires.iterrows():
    distances = np.sqrt(
        (sites['latitude'] - fire['latitude'])**2 +
        (sites['longitude'] - fire['longitude'])**2
    )

    nearest_idx.append(distances.idxmin())

nearest_idx = np.array(nearest_idx)

print("✅ Nearest industrial sites calculated!")

In [ ]:
import os
import pandas as pd

fires_path = r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data\sample_fire_data.csv"

print("Save path:", fires_path)
print("File exists:", os.path.exists(fires_path))

In [ ]:
# Nearest industrial site ka actual NAME
fires['nearest_industry'] = sites.iloc[nearest_idx]['site_name'].values

# Nearest industrial site ka TYPE
fires['industry_type'] = sites.iloc[nearest_idx]['site_type'].values

# Save
fires.to_csv(fires_path, index=False)

print("✅ Industrial site name and type successfully added!")

print(
    fires[
        ['latitude', 'longitude', 'nearest_industry', 'industry_type']
    ].head(10)
)

ML training 

In [ ]:
import os

data_path = r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data"

print(os.listdir(data_path))

In [ ]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data\processed_thermal_sources.csv"
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
import pandas as pd

data_path = r"C:\Users\Hp\OneDrive\Desktop\SIH26162\data\classified_fire_data.csv"

df_ml = pd.read_csv(data_path)

print("ML Data Shape:", df_ml.shape)

print("\nClass Distribution:")
print(df_ml["source_class"].value_counts())

In [ ]:
print("Actual columns in df_ml:")
print(df_ml.columns.tolist())

In [ ]:
print("\nShape:")
print(df_ml.shape)

print("\nFirst 5 rows:")
display(df_ml.head())

In [ ]:
print("ALL COLUMNS:")
for i, col in enumerate(df_ml.columns):
    print(i, repr(col))

print("\nSHAPE:", df_ml.shape)

In [ ]:
features = [
    "brightness",
    "confidence",
    "frp",
    "industry_type",
    "distance_to_industry_km",
    "persistence_count"
]

target = "source_class"

ml_df = df_ml[features + [target]].copy()

print("ML Data Shape:", ml_df.shape)

print("\nColumns:")
print(ml_df.columns.tolist())

print("\nClass Distribution:")
print(ml_df["source_class"].value_counts())


In [ ]:
print("Total Classes:", ml_df["source_class"].nunique())

print("\nClasses:")
for i, cls in enumerate(sorted(ml_df["source_class"].unique())):
    print(i, "=", cls)

In [ ]:
from sklearn.preprocessing import LabelEncoder

industry_encoder = LabelEncoder()
target_encoder = LabelEncoder()

ml_df["industry_type_encoded"] = industry_encoder.fit_transform(
    ml_df["industry_type"].astype(str)
)

ml_df["source_class_encoded"] = target_encoder.fit_transform(
    ml_df["source_class"].astype(str)
)

print("Industry Types:")
print(list(industry_encoder.classes_))

print("\nSource Classes:")
for i, cls in enumerate(target_encoder.classes_):
    print(i, "=", cls)

In [ ]:
X = ml_df[
    [
        "brightness",
        "confidence",
        "frp",
        "industry_type_encoded",
        "distance_to_industry_km",
        "persistence_count"
    ]
]

y = ml_df["source_class_encoded"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest Training Completed!")

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Random Forest Accuracy: {accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    target_names=target_encoder.classes_,
    zero_division=0
)

print(report)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm)

plt.xticks(
    range(len(target_encoder.classes_)),
    target_encoder.classes_,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(target_encoder.classes_)),
    target_encoder.classes_
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Random Forest Confusion Matrix")

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

display(importance)

In [ ]:
import pickle

model_path = r"C:\Users\Hp\OneDrive\Desktop\SIH26162\models\trained_model.pkl"

model_data = {
    "model": rf_model,
    "industry_encoder": industry_encoder,
    "target_encoder": target_encoder,
    "features": [
        "brightness",
        "confidence",
        "frp",
        "industry_type_encoded",
        "distance_to_industry_km",
        "persistence_count"
    ]
}

with open(model_path, "wb") as f:
    pickle.dump(model_data, f)

print("Model saved successfully!")
print(model_path)

In [ ]:
with open(model_path, "rb") as f:
    saved_model = pickle.load(f)

print("Saved Model Classes:")

for i, cls in enumerate(saved_model["target_encoder"].classes_):
    print(i, "=", cls)

print("\nTotal Classes:",
      len(saved_model["target_encoder"].classes_))